# 07 — Reading & Writing Files

Pandas supports 20+ file formats. This notebook covers CSV, Excel, JSON, SQL, Parquet,
Feather, and URLs — with every important parameter you'll actually use in production.

In [ ]:
import pandas as pd
import numpy as np
import os
import tempfile
from io import StringIO, BytesIO

# Temp directory for all file operations
TMPDIR = tempfile.mkdtemp()
print(f"Working dir: {TMPDIR}")

# Create the primary dataset
np.random.seed(42)
n = 500
employees = pd.DataFrame({
    'emp_id':           range(1001, 1001 + n),
    'name':             [f'Employee_{i:03d}' for i in range(n)],
    'department':       np.random.choice(['Engineering', 'Sales', 'HR', 'Marketing', 'Finance'], n),
    'salary':           np.random.randint(40000, 150000, n),
    'experience_years': np.random.randint(1, 20, n),
    'join_date':        pd.date_range('2010-01-01', periods=n, freq='W'),
    'rating':           np.random.choice([1.0, 2.0, 3.0, 4.0, 5.0], n),
    'city':             np.random.choice(['Mumbai', 'Delhi', 'Bangalore', 'Pune', 'Chennai'], n),
    'active':           np.random.choice([True, False], n, p=[0.85, 0.15])
})

employees.head(3)

## 1. CSV — read_csv() Most Important Parameters

In [ ]:
csv_path = os.path.join(TMPDIR, 'employees.csv')
employees.to_csv(csv_path, index=False)  # index=False: don't write row numbers
print(f"Written: {os.path.getsize(csv_path) / 1024:.1f} KB")

In [ ]:
# Basic read
df = pd.read_csv(csv_path)
print(df.dtypes)
print(f"Shape: {df.shape}")

In [ ]:
# Key parameters
df = pd.read_csv(
    csv_path,
    usecols=['emp_id', 'name', 'department', 'salary'],  # read only needed cols
    dtype={'emp_id': 'int32', 'salary': 'int32'},        # specify dtypes upfront
    parse_dates=['join_date'],                            # auto-parse dates
    nrows=100,                                           # read only N rows
)
print(df.dtypes)
print(f"Shape: {df.shape}")

In [ ]:
# Handling messy CSVs
messy_csv = """# Company Employee Export
# Generated: 2024-01-01
ID|Name|Department|Salary
1001|Alice|Engineering|90,000
1002|Bob|Sales|85,000
1003|Carol|HR|N/A
"""

df_messy = pd.read_csv(
    StringIO(messy_csv),
    sep='|',              # custom separator
    skiprows=2,           # skip comment lines
    na_values=['N/A', 'n/a', 'NULL', '-'],  # additional NA values
    thousands=',',        # remove comma in numbers like 90,000
)
print(df_messy)
print(df_messy.dtypes)

In [ ]:
# Writing CSV — important options
employees.head(10).to_csv(
    os.path.join(TMPDIR, 'sample.csv'),
    index=False,          # don't write index
    sep=',',
    encoding='utf-8',
    date_format='%Y-%m-%d',  # control datetime format in output
    float_format='%.2f',     # decimal precision
    na_rep='NULL'            # how to represent NaN
)
print("Written sample CSV")

## 2. Excel — read_excel() / to_excel()

In [ ]:
excel_path = os.path.join(TMPDIR, 'employees.xlsx')

# Write to Excel — multiple sheets
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    employees[employees['department'] == 'Engineering'].to_excel(
        writer, sheet_name='Engineering', index=False
    )
    employees[employees['department'] == 'Sales'].to_excel(
        writer, sheet_name='Sales', index=False
    )
    employees.describe().to_excel(
        writer, sheet_name='Summary'
    )

print(f"Excel file: {os.path.getsize(excel_path) / 1024:.1f} KB")

In [ ]:
# Read specific sheet
try:
    df_eng = pd.read_excel(excel_path, sheet_name='Engineering')
    print(f"Engineering sheet: {df_eng.shape}")

    # Read all sheets as dict
    all_sheets = pd.read_excel(excel_path, sheet_name=None)  # None = all sheets
    for name, df in all_sheets.items():
        print(f"Sheet '{name}': {df.shape}")

except Exception as e:
    print(f"openpyxl not installed: {e}")
    print("Install with: pip install openpyxl")

## 3. JSON — read_json() / to_json()

In [ ]:
json_path = os.path.join(TMPDIR, 'employees.json')

# orientations: 'records' (list of dicts) is most common/interoperable
employees.head(5).to_json(json_path, orient='records', indent=2, date_format='iso')

with open(json_path) as f:
    print(f.read()[:500])

In [ ]:
# Read records JSON
df_json = pd.read_json(json_path, orient='records')
print(df_json.dtypes)
print(df_json.head(3))

In [ ]:
# JSON orientations
sample = employees.head(3)[['name', 'salary', 'department']]

for orient in ['records', 'split', 'index', 'columns', 'values']:
    print(f"\n--- orient='{orient}' ---")
    print(sample.to_json(orient=orient)[:120])

## 4. SQL — read_sql() / to_sql()

In [ ]:
import sqlite3

db_path = os.path.join(TMPDIR, 'company.db')
conn = sqlite3.connect(db_path)

# Write DataFrame to SQL table
employees.to_sql(
    name='employees',
    con=conn,
    if_exists='replace',  # 'fail', 'replace', 'append'
    index=False,
    dtype={'join_date': 'TEXT'}  # SQLite has no native datetime
)
print("Table written")

In [ ]:
# Read full table
df_sql = pd.read_sql('SELECT * FROM employees LIMIT 5', conn)
print(df_sql.head())

In [ ]:
# Read with complex SQL query
query = """
    SELECT department,
           COUNT(*)       as headcount,
           AVG(salary)    as avg_salary,
           MAX(salary)    as max_salary
    FROM employees
    WHERE active = 1
    GROUP BY department
    ORDER BY avg_salary DESC
"""
dept_stats = pd.read_sql(query, conn)
print(dept_stats.round(0))

conn.close()

## 5. Parquet — The Production Standard for Analytics

In [ ]:
parquet_path = os.path.join(TMPDIR, 'employees.parquet')

try:
    # Write
    employees.to_parquet(parquet_path, engine='pyarrow', compression='snappy', index=False)

    csv_size     = os.path.getsize(csv_path)
    parquet_size = os.path.getsize(parquet_path)

    print(f"CSV size:     {csv_size / 1024:.1f} KB")
    print(f"Parquet size: {parquet_size / 1024:.1f} KB")
    print(f"Compression:  {(1 - parquet_size/csv_size):.1%}")

    # Read
    df_pq = pd.read_parquet(parquet_path)
    print(f"\nShape: {df_pq.shape}")
    print(df_pq.dtypes)

except ImportError:
    print("pyarrow not installed. Install with: pip install pyarrow")

In [ ]:
# Parquet column pruning — read only needed columns (major speedup on wide tables)
try:
    df_subset = pd.read_parquet(
        parquet_path,
        columns=['name', 'department', 'salary']  # only 3 columns read from disk
    )
    print(f"Column-pruned read: {df_subset.shape}")
    print(df_subset.head(3))
except Exception as e:
    print(f"Skipping: {e}")

## 6. Feather — Fastest for In-Process Exchange

In [ ]:
feather_path = os.path.join(TMPDIR, 'employees.feather')

try:
    import pyarrow.feather as feather
    employees.to_feather(feather_path)
    df_feather = pd.read_feather(feather_path)
    print(f"Feather read: {df_feather.shape}")
    feather_size = os.path.getsize(feather_path)
    print(f"Feather size: {feather_size / 1024:.1f} KB")
except ImportError:
    print("pyarrow not installed")

## 7. Reading from URLs

In [ ]:
# Reading CSVs directly from URLs
# Titanic dataset from a reliable public URL
TITANIC_URL = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'

try:
    titanic = pd.read_csv(TITANIC_URL)
    print(f"Titanic loaded: {titanic.shape}")
    print(titanic.head(3))
except Exception as e:
    print(f"URL read failed (no internet?): {e}")
    # Fallback: create minimal titanic-like data
    titanic = pd.DataFrame({
        'PassengerId': range(1, 6),
        'Survived': [0, 1, 1, 1, 0],
        'Pclass': [3, 1, 3, 1, 3],
        'Name': ['Braund, Mr. Owen', 'Cumings, Mrs. John', 'Heikkinen, Miss. Laina',
                 'Futrelle, Mrs. Jacques', 'Allen, Mr. William'],
        'Sex': ['male', 'female', 'female', 'female', 'male'],
        'Age': [22.0, 38.0, 26.0, 35.0, 35.0],
        'Fare': [7.25, 71.28, 7.93, 53.10, 8.05]
    })
    print("Using fallback Titanic data")
    print(titanic)

## 8. StringIO — In-Memory CSV (Testing / APIs)

In [ ]:
# Common pattern: API returns CSV text → parse in memory
api_response_text = """
product_id,product_name,category,price,stock
P001,Laptop Pro,Electronics,85999,50
P002,Desk Chair,Furniture,12999,200
P003,Python Book,Books,599,1000
P004,Webcam HD,Electronics,3999,150
""".strip()

products = pd.read_csv(
    StringIO(api_response_text),
    dtype={'price': 'int32', 'stock': 'int16'}
)
print(products)
print(products.dtypes)

## 9. Clipboard — Quick Data Import

In [ ]:
# pd.read_clipboard() — reads data you've copied from Excel/Google Sheets
# pd.DataFrame.to_clipboard() — copies to clipboard for pasting into Excel
# These are interactive — just know they exist
print("Clipboard functions:")
print("  pd.read_clipboard()              # read from clipboard")
print("  df.to_clipboard(index=False)     # copy df to clipboard")
print("Useful for quickly moving data between Jupyter and Excel/Sheets")

## 10. Format Comparison — When to Use What

| Format | Best For | Notes |
|--------|----------|-------|
| **CSV** | Human-readable exchange, simple data | No dtypes, large size |
| **Excel** | Business reporting, multiple sheets | Slow, needs openpyxl |
| **JSON** | APIs, nested data | verbose, no dtypes |
| **SQL** | Persistent storage, querying | Needs DB connection |
| **Parquet** | Analytics, production pipelines | Columnar, compressed, fast |
| **Feather** | In-process fast exchange (R↔Python) | Not for long-term storage |

**Default rule**: Use **Parquet** for all analytical data at rest.

## 11. to_csv() vs to_parquet() Speed Comparison

In [ ]:
import time

big = employees.loc[employees.index.repeat(200)].reset_index(drop=True)  # 100K rows
print(f"Test DataFrame: {big.shape}")

# Write CSV
csv_big = os.path.join(TMPDIR, 'big.csv')
start = time.perf_counter()
big.to_csv(csv_big, index=False)
t_csv_write = time.perf_counter() - start

# Read CSV
start = time.perf_counter()
_ = pd.read_csv(csv_big)
t_csv_read = time.perf_counter() - start

try:
    parquet_big = os.path.join(TMPDIR, 'big.parquet')
    
    # Write Parquet
    start = time.perf_counter()
    big.to_parquet(parquet_big, index=False)
    t_pq_write = time.perf_counter() - start
    
    # Read Parquet
    start = time.perf_counter()
    _ = pd.read_parquet(parquet_big)
    t_pq_read = time.perf_counter() - start
    
    print(f"CSV   — Write: {t_csv_write:.2f}s  Read: {t_csv_read:.2f}s  Size: {os.path.getsize(csv_big)/1024**2:.1f}MB")
    print(f"Parquet — Write: {t_pq_write:.2f}s  Read: {t_pq_read:.2f}s  Size: {os.path.getsize(parquet_big)/1024**2:.1f}MB")

except ImportError:
    print(f"CSV   — Write: {t_csv_write:.2f}s  Read: {t_csv_read:.2f}s  Size: {os.path.getsize(csv_big)/1024**2:.1f}MB")
    print("Parquet: pyarrow not installed")